# Notebook 06 (Revised) — 1D Convolutional Neural Network Training

**CMSC 190 Special Problem**
*Classification of High-Protein and Low-Protein Corn (Zea Mays) Using NIR Spectral Data and Machine Learning Techniques*

---

This notebook builds, trains, and evaluates a **1D Convolutional Neural Network (1D-CNN)** on the preprocessed and augmented NIR spectral data produced by the revised notebooks.

| Notebook | What it did |
|---|---|
| 01 | Loaded and explored the raw NIR data |
| 02 | Assigned High / Low protein binary labels |
| 03 | Applied Savitzky-Golay smoothing |
| 04 (revised) | All 80 originals → test set; generated 1936 synthetic training samples |
| 05 (revised) | Trained and evaluated PLS-DA and SVM on the revised data |
| **06 revised (this notebook)** | **Trains and evaluates the 1D-CNN on the revised data** |

| Notebook | Strategy |
|---|---|
| `06_1d_cnn_training.ipynb` (original) | Trained on 64 originals + augmented copies; tested on 16 samples |
| **`06_1d_cnn_revised.ipynb` (this notebook)** | **Trained on 1936 synthetic-only samples; tested on all 80 originals** |

**Output:** `saved_models/revised/1d_cnn_best.h5`

## Section 1 — Imports and Setup

This cell imports all the libraries we need and sets random seeds to make sure our results are the same every time the notebook is run.

We also import the helper functions from `src/cnn_trainer.py`, which contains all the CNN-specific logic for this notebook.

> **Note:** `scale_for_cnn` and `train_cnn` from `src/cnn_trainer.py` are **not** imported here because they hardcode save paths relative to `notebooks/` (i.e. `../saved_models/`). Since this notebook lives in `notebooks/revised/`, those two steps are inlined in Sections 2 and 5 respectively, using `../../saved_models/revised/` instead. All other helper functions are imported as normal.

> **Key advantage of 1D-CNN over PLS-DA and SVM:**
> The 1D-CNN **automatically learns which spectral features matter** directly from the raw wavelength data during backpropagation — no manual feature engineering (like PLS components or kernel tricks) is needed. The convolutional filters act as learnable pattern detectors that slide across the spectrum, capturing local absorption relationships that human-designed features might miss.

In [ ]:
import importlib
import os
import random
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Allow Python to find the src/ package two directories above notebooks/revised/
sys.path.append('../..')

# Force Python to re-read src/ files from disk instead of using any
# cached (stale) module version from previous kernel runs.
for _mod in list(sys.modules.keys()):
    if _mod.startswith('src.'):
        del sys.modules[_mod]
importlib.invalidate_caches()

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- Reproducibility seeds (set BEFORE any other TF/Keras calls) ---
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# --- Custom helper functions (path-independent ones only) ---
from src.cnn_trainer import (
    build_1d_cnn,
    evaluate_cnn,
    plot_cnn_confusion_matrix,
    plot_training_history,
    reshape_for_cnn,
)
from src.trainer import save_model

# --- Plot style ---
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print("All imports successful.")
print(f"TensorFlow version : {tf.__version__}")

## Section 2 — Load Data

Here we load the four NumPy arrays that were produced by the revised notebook 04.

- **`X_train_synthetic.npy`** — 1936 synthetic training spectra (shape: 1936 × 700 wavelengths)
- **`y_train_synthetic.npy`** — labels for the synthetic training set
- **`X_test_all.npy`** — all 80 original spectra, reserved exclusively as the test set
- **`y_test_all.npy`** — labels for the all-80 test set

> **Important:** `X_test` and `y_test` are **only used in Section 7** for final evaluation.
> They are loaded here purely to confirm the shapes are correct.
> We will NOT modify, augment, or preprocess them further.

In [ ]:
# Load data from the revised processed folder.
# Paths are relative to the notebooks/revised/ directory.
X_train = np.load('../../data/processed/revised/X_train_synthetic.npy')
y_train = np.load('../../data/processed/revised/y_train_synthetic.npy')
X_test  = np.load('../../data/processed/revised/X_test_all.npy')
y_test  = np.load('../../data/processed/revised/y_test_all.npy')

print("=== Array Shapes ===")
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}")

print("\n=== Class Distribution (y_train) ===")
classes, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(classes, counts):
    label = 'High Protein' if cls == 1 else 'Low Protein'
    print(f"  Class {int(cls)} ({label}): {cnt} samples")

print("\n=== Class Distribution (y_test) ===")
classes_t, counts_t = np.unique(y_test, return_counts=True)
for cls, cnt in zip(classes_t, counts_t):
    label = 'High Protein' if cls == 1 else 'Low Protein'
    print(f"  Class {int(cls)} ({label}): {cnt} samples")

## Scaling the Input Data

Before feeding the spectra into the CNN, we apply **StandardScaler** normalization.

This transforms each of the 700 wavelength channels so that it has **mean = 0** and **standard deviation = 1** across all training samples. CNNs are sensitive to input scale — without normalization, the gradients during backpropagation become very uneven across wavelengths, which caused the model to get stuck at ~50% accuracy (random guessing).

**Why fit only on `X_train`?**
The scaler learns the mean and std from the training set only. Applying those same parameters to `X_test` ensures we do not leak any information from the test set into the model — a form of *data-leakage prevention*.

The fitted scaler is saved to `saved_models/revised/cnn_scaler.pkl` for use during inference.

> **Note:** This step is inlined here (rather than using `scale_for_cnn` from `src/cnn_trainer.py`) because that helper hardcodes a save path relative to `notebooks/`. The logic is identical.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

os.makedirs('../../saved_models/revised', exist_ok=True)
joblib.dump(scaler, '../../saved_models/revised/cnn_scaler.pkl')

print('CNN scaler saved to ../../saved_models/revised/cnn_scaler.pkl')
print(f'X_train_scaled shape: {X_train_scaled.shape}')
print(f'X_test_scaled  shape: {X_test_scaled.shape}')

## Section 3 — Reshape Data for CNN Input

Before feeding the spectra into the 1D-CNN, we need to add an extra dimension.

**Why?**
Keras `Conv1D` expects its input to have shape `(samples, timesteps, features)` — a 3D tensor.

- **`samples`** — the number of spectra (rows)
- **`timesteps`** — the number of wavelength points (700 in our case)
- **`features`** — the number of values at each timestep (1 here, because each wavelength has exactly one absorbance reading)

Think of the 700-point NIR spectrum as a time series where each "moment in time" is a single wavelength, and the "signal value" at that moment is the absorbance. The `Conv1D` filters then slide along this sequence, learning to recognise absorption patterns across neighboring wavelengths.

`reshape_for_cnn()` handles this transformation automatically.

In [ ]:
# Reshape the SCALED splits from (n_samples, 700) → (n_samples, 700, 1)
# so they are compatible with the Conv1D input layer.
X_train_cnn = reshape_for_cnn(X_train_scaled)
X_test_cnn  = reshape_for_cnn(X_test_scaled)

print("Shapes after reshaping:")
print(f"  X_train_cnn : {X_train_cnn.shape}  (expected: (n_samples, 700, 1))")
print(f"  X_test_cnn  : {X_test_cnn.shape}  (expected: (n_samples, 700, 1))")

## Section 4 — Build the 1D-CNN Model

This cell creates the neural network architecture. The `build_1d_cnn()` function returns a compiled Keras model and prints a layer-by-layer summary.

Here is what each layer does in plain English:

| Layer | Purpose |
|---|---|
| **Conv1D (64 filters, kernel 7)** | Scans the spectrum with 64 small windows of width 7 wavelengths each. Each filter learns to recognise a different local absorption pattern. |
| **MaxPooling1D (pool 2)** | Halves the sequence length by keeping only the strongest activation in each pair of neighboring positions. Reduces computation and focuses on the most prominent features. |
| **Conv1D (128 filters, kernel 5)** | A second, deeper scanning layer that combines the patterns found by the first layer into 128 higher-level spectral features. |
| **MaxPooling1D (pool 2)** | Halves the sequence length again, further compressing the representation. |
| **Flatten** | Converts the 2D feature map (sequence × filters) into a single long 1D vector so it can be fed into Dense layers. |
| **Dense (64 units, relu)** | A fully-connected layer that learns non-linear combinations of all the extracted spectral features. |
| **Dropout (rate 0.3)** | Randomly turns off 30% of the neurons during every training step. This forces the network to learn redundant representations and prevents overfitting. |
| **Dense (32 units, relu)** | A second fully-connected layer that refines the feature combinations. |
| **Dense (1 unit, sigmoid)** | The output neuron. Sigmoid squashes the output to a value between 0 and 1. Values **≥ 0.5** are classified as **High Protein (1)**; values **< 0.5** are **Low Protein (0)**. |

In [ ]:
# Build and compile the 1D-CNN.
# The function prints the full model summary automatically.
model = build_1d_cnn(input_length=700)

## Section 5 — Train the Model

This cell runs the actual training loop. Two safety mechanisms are active throughout:

**EarlyStopping**
- Monitors `val_loss` (the loss on the 10% validation slice) after every epoch.
- If `val_loss` does not improve for **20 consecutive epochs**, training stops automatically.
- The model is then rolled back to the weights that achieved the best `val_loss` (`restore_best_weights=True`).
- This prevents the model from continuing to memorise the training data long after it has stopped generalising.

**ModelCheckpoint**
- After every epoch, if `val_loss` is the lowest seen so far, the model is saved to `saved_models/revised/1d_cnn_best.h5`.
- Only the single best checkpoint is kept (`save_best_only=True`).
- This guarantees that even if training is interrupted, the best weights are preserved on disk.

**Data split during training:**
The 90 / 10 split of `X_train_cnn` happens internally inside Keras — **the 10% held-out validation samples are a random subset of the training data, not the test set.**

> **Note:** This cell inlines the callback setup (instead of calling `train_cnn` from `src/cnn_trainer.py`) because that function hardcodes `../saved_models/1d_cnn_best.h5`. The logic is identical; only the checkpoint path differs.

In [ ]:
tf.random.set_seed(42)

os.makedirs('../../saved_models/revised', exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath='../../saved_models/revised/1d_cnn_best.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    X_train_cnn,
    y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1,
)

epochs_run = len(history.history['loss'])
print(f"\nTraining complete. Ran {epochs_run} epoch(s).")

## Section 6 — Plot Training History

The two charts below show how the model's performance changed over each epoch of training.

**What to look for:**

- **Healthy training** — both the training loss and validation loss decrease steadily together, and both accuracy curves rise together. This means the model is learning genuine spectral patterns, not just memorising the training data.

- **Overfitting** — if the training loss keeps going down but the validation loss flattens or rises, the model has started memorising the training examples instead of generalising. The gap between the two curves is the "overfitting gap".

- **EarlyStopping at work** — if training stopped before epoch 100, the vertical position where the curves end shows how many epochs were actually needed. The saved model uses the weights from the epoch with the lowest validation loss (the best point on the `val_loss` curve).

In [ ]:
# Plot the loss and accuracy curves recorded during training.
plot_training_history(history)

## Section 7 — Evaluate on Test Set

> **This is the first and only time `X_test` and `y_test` are used.**

We now run the trained model on the **all 80 original** held-out test samples that were reserved by the revised notebook 04 and have not been seen during training, validation, or hyperparameter tuning.

The metrics reported here (Accuracy, Precision, Recall, F1-Score) are the **official final metrics for the 1D-CNN**. They will be carried forward to the revised notebook 07 for the head-to-head comparison against PLS-DA and SVM.

The confusion matrix shows:
- **True Positives** (top-left): High-Protein samples correctly identified as High Protein
- **True Negatives** (bottom-right): Low-Protein samples correctly identified as Low Protein
- **False Positives** (top-right): Low-Protein samples incorrectly predicted as High Protein
- **False Negatives** (bottom-left): High-Protein samples incorrectly predicted as Low Protein

In [ ]:
# Evaluate the trained model on the held-out test set.
# Predictions use threshold=0.5: sigmoid >= 0.5 → High Protein (1)
results_cnn = evaluate_cnn(model, X_test_cnn, y_test)

# Plot the confusion matrix for the test-set predictions.
plot_cnn_confusion_matrix(y_test, results_cnn['y_pred'])

## Section 8 — Confirm Model Was Saved

The best model weights were already saved **automatically** to `saved_models/revised/1d_cnn_best.h5` by the `ModelCheckpoint` callback during training. No manual saving is needed.

The cell below simply confirms that the file is present on disk.

> **Note for the revised Notebook 07:** When building the final comparison, it will load:
> - `saved_models/revised/pls_da_best.pkl` — best PLS-DA model (saved in revised Notebook 05)
> - `saved_models/revised/svm_best.pkl`    — best SVM model (saved in revised Notebook 05)
> - `saved_models/revised/1d_cnn_best.h5`  — best 1D-CNN model (saved in this notebook)
>
> All three models will be evaluated on the same `X_test_all` (80 original samples) for a fair, controlled comparison.

In [ ]:
path = '../../saved_models/revised/1d_cnn_best.h5'

if os.path.exists(path):
    size_kb = os.path.getsize(path) / 1024
    print(f"Model saved successfully: {path}  ({size_kb:.1f} KB)")
else:
    print("WARNING: Model file not found. Check training output.")

## Section 9 — Summary

This notebook trained and evaluated a 1D Convolutional Neural Network on NIR spectral data for binary protein classification of corn samples using the **revised pipeline**.

---

### Model Architecture

```
Input  →  Conv1D(64, k=7, relu, same)  →  MaxPool(2)
       →  Conv1D(128, k=5, relu, same) →  MaxPool(2)
       →  Flatten
       →  Dense(64, relu)  →  Dropout(0.3)
       →  Dense(32, relu)
       →  Dense(1, sigmoid)
```

### Input Shape
`(n_samples, 700, 1)` — 700 NIR wavelength points per sample, treated as a 1D sequence

### Training Configuration

| Setting | Value |
|---|---|
| Optimizer | Adam (lr = 0.001) |
| Loss function | Binary cross-entropy |
| Maximum epochs | 100 |
| Batch size | 64 |
| Validation split | 10% of training data |

### Regularization Strategy

| Technique | Configuration |
|---|---|
| EarlyStopping | Monitors `val_loss`, patience = 20, restores best weights |
| Dropout | Rate = 0.3 (applied after first Dense layer) |

### Output Artifact
`saved_models/revised/1d_cnn_best.h5` — Keras model file holding the best weights found during training

### Evaluation Metrics (on held-out test set)
Accuracy, Precision, Recall, and F1-Score were computed on all 80 original spectra — data the model has never seen before.

---

> **Next step — Revised Notebook 07:** All three trained models (PLS-DA, SVM, and 1D-CNN) will be loaded from `saved_models/revised/` and evaluated side-by-side on the same 80-sample test set to determine which approach best classifies High-Protein and Low-Protein corn from NIR spectra.